# Predicting default on credit card payments

- A Binary Classification using SageMaker

src: https://aws.amazon.com/blogs/machine-learning/simplify-machine-learning-with-xgboost-and-amazon-sagemaker/

---

### 📝 Problem Statement

The objective of this project is to build a predictive model that identifies whether a credit card holder is likely to default on their payment in the upcoming month. By analyzing historical customer attributes—such as demographic information, past payment behavior, and billing history—we aim to classify each individual as a potential **defaulter (1)** or **non-defaulter (0)**.

Accurate predictions of customer default risk enable financial institutions to:

* Assess customer creditworthiness more effectively,
* Make data-driven credit issuance decisions,
* Proactively manage risk,
* And forecast future payment behaviors.

The model is trained using XGBoost on structured tabular data and deployed to an Amazon SageMaker real-time inference endpoint for scalable, production-ready predictions.

---

In [ ]:
bucket = 'yourname-sagemaker'
prefix = 'sagemaker/xgboost_credit_risk'

# Define IAM role
# import boto3
# import re
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# import os
import sagemaker
from sagemaker import get_execution_role
from sagemaker.inputs import TrainingInput
from sagemaker.serializers import CSVSerializer

role = get_execution_role()

In [5]:
!pip install boto3

In [4]:
import boto3
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

## Download the dataset

In [1]:
!wget https://archive.ics.uci.edu/ml/machine-learning-databases/00350/default%20of%20credit%20card%20clients.xls

--2025-07-11 01:38:31--  https://archive.ics.uci.edu/ml/machine-learning-databases/00350/default%20of%20credit%20card%20clients.xls
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘default of credit card clients.xls’

default of credit c     [   <=>              ]   5.28M  9.74MB/s    in 0.5s    

2025-07-11 01:38:32 (9.74 MB/s) - ‘default of credit card clients.xls’ saved [5539328]



In [6]:
dataset = pd.read_excel('default of credit card clients.xls')
pd.set_option('display.max_rows', 8)
pd.set_option('display.max_columns', 15)
dataset

,Unnamed: 0,X1,X2,X3,X4,X5,X6,...,X18,X19,X20,X21,X22,X23,Y
0,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,...,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
1,1,20000,2,2,1,24,2,...,0,689,0,0,0,0,1
2,2,120000,2,2,2,26,-1,...,0,1000,1000,1000,0,2000,1
3,3,90000,2,2,2,34,0,...,1518,1500,1000,1000,1000,5000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29997,29997,150000,1,3,2,43,-1,...,1837,3526,8998,129,0,0,0
29998,29998,30000,1,2,2,37,4,...,0,0,22000,4200,2000,3100,1
29999,29999,80000,1,3,1,41,1,...,85900,3409,1178,1926,52964,1804,1
30000,30000,50000,1,2,1,46,0,...,2078,1800,1430,1000,1000,1000,1


Here's a clean and beginner-friendly formatted version of your dataset description:

---

### 📊 Dataset Overview

* **Total Records**: 30,000
* **Total Attributes per Record**: 23
* **Goal**: Predict whether a person will default on credit payment (target variable = `Y`)

---

### 🔢 Feature Descriptions

| **Feature** | **Description**                                                                                             |
| ----------- | ----------------------------------------------------------------------------------------------------------- |
| `X1`        | Amount of the given credit (includes individual and family/supplementary credit).                           |
| `X2`        | Gender: `1` = Male, `2` = Female.                                                                           |
| `X3`        | Education level: <br> `1` = Graduate school <br> `2` = University <br> `3` = High school <br> `4` = Others. |
| `X4`        | Marital status: <br> `1` = Married <br> `2` = Single <br> `3` = Others.                                     |
| `X5`        | Age (in years).                                                                                             |

---

### 📆 History of Past Payments (`X6` – `X11`)

These features track the repayment status over the past 6 months (from April to September 2005):

| Feature | Month    | Description      |
| ------- | -------- | ---------------- |
| `X6`    | Sep 2005 | Repayment status |
| `X7`    | Aug 2005 | Repayment status |
| `X8`    | Jul 2005 | Repayment status |
| `X9`    | Jun 2005 | Repayment status |
| `X10`   | May 2005 | Repayment status |
| `X11`   | Apr 2005 | Repayment status |

**Repayment Status Scale**:

* `-1`: Paid on time
* `1`: Delay for 1 month
* `2`: Delay for 2 months
  …
* `8`: Delay for 8 months
* `9`: Delay for 9 months or more

---

### 💳 Bill Statement Amounts (`X12` – `X17`)

| Feature | Month    | Description              |
| ------- | -------- | ------------------------ |
| `X12`   | Sep 2005 | Amount of bill statement |
| `X13`   | Aug 2005 | Amount of bill statement |
| `X14`   | Jul 2005 | Amount of bill statement |
| `X15`   | Jun 2005 | Amount of bill statement |
| `X16`   | May 2005 | Amount of bill statement |
| `X17`   | Apr 2005 | Amount of bill statement |

---

### 💸 Amount Paid in Previous Months (`X18` – `X23`)

| Feature | Month    | Description |
| ------- | -------- | ----------- |
| `X18`   | Sep 2005 | Amount paid |
| `X19`   | Aug 2005 | Amount paid |
| `X20`   | Jul 2005 | Amount paid |
| `X21`   | Jun 2005 | Amount paid |
| `X22`   | May 2005 | Amount paid |
| `X23`   | Apr 2005 | Amount paid |

---

### 🎯 Target Variable

| Feature | Description                                                                                                       |
| ------- | ----------------------------------------------------------------------------------------------------------------- |
| `Y`     | Did the person default on payment the following month? <br> `1` = Yes (defaulted) <br> `0` = No (did not default) |

---


---

### 🔧 Feature Engineering (Skipped in This Tutorial)

In a typical machine learning workflow, **feature engineering** is a critical step. It involves:

* **Selecting the best input features** for the model
* **Transforming raw data** into meaningful signals
* **Creating new features** or modifying existing ones to improve performance

🌀 **Note:** Feature engineering is **iterative** — you often:

* Try different feature combinations
* Train multiple models
* Evaluate and refine until you achieve the best performance

---

🚧 **However, for this tutorial**:

To keep things simple and focus on using **XGBoost**, we will **skip feature engineering** and
✅ Train the model **using the raw features exactly as provided** in the dataset.

---

## Format the dataset

To format our dataset for this, we drop the “ID” column that numbers the rows in the first column and we modify the “Y” column to be the first column in our DataFrame.

In [7]:
# Step 1: Remove the unnecessary index column

# When reading a CSV file, sometimes an extra column like 'Unnamed: 0' is added automatically
# if the index from the file is also read as a column.
# This column doesn't contain useful data, so we drop it.
dataset = dataset.drop('Unnamed: 0', axis=1)
dataset

,X1,X2,X3,X4,X5,X6,X7,...,X18,X19,X20,X21,X22,X23,Y
0,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,...,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
1,20000,2,2,1,24,2,2,...,0,689,0,0,0,0,1
2,120000,2,2,2,26,-1,2,...,0,1000,1000,1000,0,2000,1
3,90000,2,2,2,34,0,0,...,1518,1500,1000,1000,1000,5000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29997,150000,1,3,2,43,-1,-1,...,1837,3526,8998,129,0,0,0
29998,30000,1,2,2,37,4,3,...,0,0,22000,4200,2000,3100,1
29999,80000,1,3,1,41,1,-1,...,85900,3409,1178,1926,52964,1804,1
30000,50000,1,2,1,46,0,0,...,2078,1800,1430,1000,1000,1000,1


In [8]:
# Step 2: Move the target column 'Y' to the front of the dataset

# 'Y' is the target variable (e.g., did the customer default or not)
# By default, it might be located at the end of the dataset.
# Here, we move it to the front to make it easier to view and work with.

# Explanation of what this line does:
# - dataset['Y']: selects the target column
# - dataset.drop(['Y'], axis=1): selects all columns except 'Y'
# - pd.concat([...], axis=1): joins them side by side (column-wise), putting 'Y' first
dataset = pd.concat([dataset['Y'], dataset.drop(['Y'], axis=1)], axis=1) # PJ: This is not mandatory; just for easy of viewing
dataset

,Y,X1,X2,X3,X4,X5,X6,...,X17,X18,X19,X20,X21,X22,X23
0,default payment next month,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,...,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6
1,1,20000,2,2,1,24,2,...,0,0,689,0,0,0,0
2,1,120000,2,2,2,26,-1,...,3261,0,1000,1000,1000,0,2000
3,0,90000,2,2,2,34,0,...,15549,1518,1500,1000,1000,1000,5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29997,0,150000,1,3,2,43,-1,...,0,1837,3526,8998,129,0,0
29998,1,30000,1,2,2,37,4,...,19357,0,0,22000,4200,2000,3100
29999,1,80000,1,3,1,41,1,...,48944,85900,3409,1178,1926,52964,1804
30000,1,50000,1,2,1,46,0,...,15313,2078,1800,1430,1000,1000,1000


Amazon SageMaker XGBoost can train on data in either a **CSV** or **LibSVM** format

### Split the dataset

We split our dataset into a training, validation, and testing set. XGBoost will train on the training dataset and use the validation set as data to evaluate prediction results as the model is trained. We will make predictions against the testing set after the model has been deployed.

In [9]:
# Step 1: Shuffle the dataset randomly
# ------------------------------------
# `dataset.sample(frac=1)` randomly shuffles the entire DataFrame.
# `frac=1` means we're keeping 100% of the rows (just shuffled).
# `random_state=1729` sets a fixed seed so results are reproducible each time you run it.
shuffled_dataset = dataset.sample(frac=1, random_state=1729)

# Step 2: Split the dataset into train, validation, and test sets
# ---------------------------------------------------------------
# We're using numpy's np.split() to break the shuffled dataset into:
# - 70% for training
# - 20% for validation
# - 10% for testing

train_data, validation_data, test_data = np.split(
    shuffled_dataset,
    [int(0.7 * len(dataset)), int(0.9 * len(dataset))]  # Split at 70% and 90% index positions
)

# Step 3: Save the training and validation sets to CSV files
# -----------------------------------------------------------
# These files will be used to train and tune the model later.
# `header=False` means we don't write column names.
# `index=False` means we don't write row numbers to the file.
train_data.to_csv('train.csv', header=False, index=False)
validation_data.to_csv('validation.csv', header=False, index=False)

/usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


---

### 📌 Summary for Beginners

| Step           | Purpose                                                                                                |
| -------------- | ------------------------------------------------------------------------------------------------------ |
| ✅ Shuffle      | Ensures that the data is randomized before splitting (avoids accidental patterns from original order). |
| ✂️ Split       | Divides the dataset into: <br> 70% training <br> 20% validation <br> 10% test.                         |
| 💾 Save to CSV | Prepares data files for use in model training and evaluation.                                          |

---

### ⚠️ Notes

* `test_data` is created but **not saved** in this snippet. If needed, add:

  ```python
  test_data.to_csv('test.csv', header=False, index=False)
  ```
* If you want column headers in the CSV (for readability), change `header=False` to `header=True`.

---


### **Step 3: Upload Train/Validation CSVs to S3 and Prepare SageMaker Inputs**

Now that our dataset is ready, we can upload it to Amazon S3 and take a note of its location so that we can use it for training.

In [ ]:
# Upload the training CSV file to S3 under the path: s3://<bucket>/<prefix>/train/train.csv
# - boto3.Session(): creates a new AWS session using your credentials
# - .resource('s3'): gets the S3 service resource
# - .Bucket(bucket): specifies which S3 bucket to use
# - .Object(...).upload_file(...): uploads a local file to the specified S3 path (key)
boto3.Session().resource('s3').Bucket(bucket).Object(os.path.join(prefix, 'train/train.csv')).upload_file('train.csv')

# Upload the validation CSV file to S3 under the path: s3://<bucket>/<prefix>/validation/validation.csv
boto3.Session().resource('s3').Bucket(bucket).Object(os.path.join(prefix, 'validation/validation.csv')).upload_file('validation.csv')

# Define the S3 input location for training data for SageMaker
# - 's3_data': path to the training data folder in S3 (not just the file)
# - 'content_type': tells SageMaker that the input format is CSV
s3_input_train = TrainingInput(
    s3_data='s3://{}/{}/train'.format(bucket, prefix),
    content_type='csv'
)

# Define the S3 input location for validation data for SageMaker
s3_input_validation = TrainingInput(
    s3_data='s3://{}/{}/validation/'.format(bucket, prefix),
    content_type='csv'
)


### **Step 4: Set Up XGBoost Estimator in SageMaker**

We first define the location of the Amazon SageMaker XGBoost training containers

We then create an Amazon SageMaker estimator.

By simply changing values such as instance_count and instance_type, we can change the size and number of instances we want to run on, which scales and distributes the training.

In [ ]:
# A dictionary of official XGBoost container URIs for different AWS regions.
# These container images are prebuilt with XGBoost and hosted in Amazon ECR (Elastic Container Registry).
# Each region must use the correct container to launch training jobs.
containers = {
    'us-west-2': '433757028032.dkr.ecr.us-west-2.amazonaws.com/xgboost:latest',
    'us-east-1': '811284229777.dkr.ecr.us-east-1.amazonaws.com/xgboost:latest',
    'us-east-2': '825641698319.dkr.ecr.us-east-2.amazonaws.com/xgboost:latest',
    'eu-west-1': '685385470294.dkr.ecr.eu-west-1.amazonaws.com/xgboost:latest'
}

# Create a SageMaker session object. This manages interactions with S3, training jobs, etc.
sess = sagemaker.Session()

# Create an XGBoost Estimator — this defines how the model will be trained in SageMaker.
xgb = sagemaker.estimator.Estimator(
    # Select the correct XGBoost container image for your current AWS region
    containers[boto3.Session().region_name],

    # IAM role that SageMaker uses to access S3 and perform training
    role,

    # Number of instances to run the training job on (1 is fine for most use cases)
    instance_count=1,

    # Type of instance to use (ml.m4.xlarge is a general-purpose training instance)
    instance_type='ml.m4.xlarge',

    # Where to store the trained model output in S3
    output_path='s3://{}/{}/output'.format(bucket, prefix),

    # Pass the SageMaker session we created earlier
    sagemaker_session=sess
)

### **Step 5: Set Hyperparameters and Launch the Training Job**

This step kicks off our XGBoost training job.

XGBoost also has a number of hyperparameters that we can tune to improve model performance. Here, we set values for some of the most commonly tuned hyperparameters. Notice the objective parameter is set to binary:logistic. This parameter tells XGBoost what kind of problem we are solving (classification, regression, ranking, etc.). In this case we are solving a binary classification proble –predicting whether or not a person is likely to default on their credit card payments.

In [ ]:
# Set the hyperparameters for the XGBoost training job:
# - eta: the learning rate (how much the model adjusts at each step)
# - objective: defines the type of learning task; here, 'binary:logistic' is used for binary classification
# - num_round: number of boosting rounds (i.e., number of trees to build)
xgb.set_hyperparameters(
    eta=0.1,                       # Learning rate
    objective='binary:logistic',   # Binary classification with logistic regression output
    num_round=25                   # Number of boosting rounds (trees)
)

# Start the training job using the input data stored in S3
# - Pass a dictionary with 'train' and 'validation' channels
# - The Estimator will automatically download this data, train the model, and save the output to S3
xgb.fit({
    'train': s3_input_train,
    'validation': s3_input_validation
})


**Training Logs**

When the training is complete, we’ll see a log of the training. This log contains metrics, such as our training and validation errors and helps us gauge the performance of our model. The training logs are also available in Amazon CloudWatch Logs.

### **Step 6: Deploy the Trained XGBoost Model as an Endpoint**

After the model is trained, this deploys our model to Amazon SageMaker hosting. It takes a few minutes to set up the hosting endpoint.

In [ ]:
# Deploy the trained model to a SageMaker endpoint so it can be used for real-time predictions.
# - initial_instance_count: number of instances to serve the endpoint (1 is fine for most use cases)
# - instance_type: type of instance used to host the endpoint
# - serializer: converts input data (e.g., a NumPy array or DataFrame row) into CSV format, which XGBoost expects

xgb_predictor = xgb.deploy(
    initial_instance_count=1,          # Number of instances in the endpoint
    instance_type='ml.m4.xlarge',      # Type of instance to deploy to
    serializer=CSVSerializer()         # sets up the serializer - Converts input data to CSV before sending to endpoint
)


### **Other Topics to explore**

1.   Batch transform instead of real-time inference,
2.   Input/output examples,
3.   Or automatic shutdown (xgb_predictor.delete_endpoint()) after testing.

### **Step 7: Use the Deployed Model to Make Predictions on Test Data**

In [ ]:
# Define a function to make predictions in batches (to avoid sending too much data at once)
def predict(data, rows=500):
    # Split the data into chunks of 'rows' size
    # This prevents memory issues when sending large datasets to the endpoint
    split_array = np.array_split(data, int(data.shape[0] / float(rows) + 1))

    predictions = ''  # Initialize an empty string to collect predictions

    # Loop over each chunk and get predictions from the deployed model
    for array in split_array:
        # xgb_predictor.predict(array) returns byte string, so we decode it to text
        # Join all prediction outputs into a single comma-separated string
        predictions = ','.join([
            predictions,
            xgb_predictor.predict(array).decode('utf-8')  # decode from bytes to string
        ])

    # Convert the comma-separated string into a NumPy array of float values
    # predictions[1:] skips the leading comma
    return np.fromstring(predictions[1:], sep=',')

# Use the function to predict on test data
# This takes numpy arrays (like our testing data) and serializes them to CSV format.
# test_data.to_numpy()[:, 1:] selects all columns except the first (which is likely the target `Y`)
predictions = predict(test_data.to_numpy()[:, 1:])

# Show predictions
predictions

# Out[97]: array([0.59283876, 0.1414548, 0.311579986, ..., 0.41416201, 0.11323805, 0.11232713])

The output is an array with our predictions. The first element of the array is the probability that the person corresponding to the first inputted data row will default on their credit card payments. The array continues with probabilities for all rows in our testing set.

### **Step 8: Convert Prediction Scores to Binary Class Labels**

In [ ]:
# Step 8: Convert Prediction Scores to Binary Class Labels
# ---------------------------------------------------------

# Define a function to make predictions in batches and return binary class labels
def predict(data, rows=500, threshold=0.5):
    # Split the data into chunks to avoid memory or API limits
    split_array = np.array_split(data, int(data.shape[0] / float(rows) + 1))

    predictions = ''  # Initialize an empty string to accumulate prediction results

    # Loop through each chunk and get predictions from the endpoint
    for array in split_array:
        # Decode the byte output from the model into a plain string
        predictions = ','.join([
            predictions,
            xgb_predictor.predict(array).decode('utf-8')
        ])

    # Convert the full comma-separated string into a NumPy array of float scores
    scores = np.fromstring(predictions[1:], sep=',')

    # Apply threshold to convert scores to binary predictions:
    # - If score > threshold → 1 (positive class)
    # - Else → 0 (negative class)
    class_labels = (scores > threshold).astype(int)

    return class_labels
    # return scores, class_labels # You can also return `scores` if you need both

# Run predictions on test data and convert to class labels
predicted_labels = predict(test_data.to_numpy()[:, 1:])
# scores, labels = predict(...)

# Show predicted labels (0 or 1)
predicted_labels


---

### 🧠 What This Step Adds:

* Converts probabilities (e.g., `0.87`, `0.12`) into class labels (e.g., `1`, `0`).
* Uses a default threshold of **0.5**, but you can change it (e.g., `threshold=0.3`) to tune sensitivity.

---

## **Clean up**

In [ ]:
xgb_predictor.delete_endpoint()